In [8]:
# ==========================================================
# 🤖 AI TEXT SUMMARIZER — PREMIUM UI
# Hugging Face T5 + Gradio + Custom CSS
# ==========================================================

!pip install -q "transformers<5.0.0" gradio sentencepiece

import gradio as gr
import torch
import warnings

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# ----------------------------------------------------------
# 1. LOAD THE MODEL DIRECTLY FROM HUGGING FACE
# ----------------------------------------------------------

MODEL_NAME = "google-t5/t5-small"

print("⏳ Downloading the Hugging Face model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()

print("✅ Hugging Face model loaded successfully!")

# ----------------------------------------------------------
# 2. SUMMARIZATION FUNCTION
# ----------------------------------------------------------

def summarize(text, max_tokens=50):

    if not text or not text.strip():
        return "Please enter some text to summarize."

    inputs = tokenizer(
        "summarize: " + text.strip(),
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            min_new_tokens=5,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    ).strip()


# ----------------------------------------------------------
# 3. APP FUNCTION
# ----------------------------------------------------------

def summarize_app(text, max_tokens):

    if not text or not text.strip():
        return (
            "✨ Your summary will appear here.",
            "0 words → 0 words",
            "Enter some text to begin."
        )

    original_words = len(text.split())

    if original_words < 20:
        return (
            "Please enter at least 20 words.",
            f"{original_words} original words",
            "More text is needed."
        )

    result = summarize(text, max_tokens)

    summary_words = len(result.split())

    reduction = round(
        (1 - summary_words / original_words) * 100
    )

    stats = (
        f"{original_words} original words  →  "
        f"{summary_words} summary words"
    )

    message = f"✨ Text reduced by approximately {reduction}%"

    return result, stats, message


# ----------------------------------------------------------
# 4. CUSTOM CSS — PREMIUM UI/UX
# ----------------------------------------------------------

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

.gradio-container {
    font-family: 'Inter', sans-serif !important;
    background:
        radial-gradient(ellipse at 10% 0%, #e4ddff 0%, transparent 38%),
        radial-gradient(ellipse at 95% 15%, #dceaff 0%, transparent 35%),
        linear-gradient(135deg, #f9faff, #f0f4ff) !important;
    max-width: 1250px !important;
    margin: auto !important;
}

#hero {
    text-align: center;
    padding: 36px 24px 28px;
    border-radius: 25px;
    background: linear-gradient(120deg, #171c4b, #44328d, #314b9d);
    color: white;
    margin: 14px 0 25px;
    box-shadow: 0 16px 40px rgba(52, 48, 125, 0.20);
}

#hero h1 {
    color: white !important;
    font-size: clamp(30px, 4vw, 46px) !important;
    font-weight: 800 !important;
    letter-spacing: -1.5px;
    margin-bottom: 12px;
}

#hero p {
    color: #e2e7ff !important;
    font-size: 16px;
    line-height: 1.8;
}

#model-tag {
    display: inline-block;
    padding: 8px 17px;
    border: 1px solid rgba(255,255,255,.3);
    border-radius: 50px;
    background: rgba(255,255,255,.10);
    color: #ffffff;
    font-size: 13px;
    margin-top: 8px;
}

.panel {
    background: rgba(255,255,255,.92);
    border: 1px solid #e1e6ff;
    border-radius: 22px;
    padding: 22px !important;
    box-shadow: 0 10px 30px rgba(49, 61, 125, .07);
}

.panel h3 {
    color: #202b63 !important;
}

textarea, input {
    border-radius: 14px !important;
    border: 1px solid #d8def5 !important;
    background: #fbfcff !important;
}

textarea:focus {
    border-color: #8a63f6 !important;
    box-shadow: 0 0 0 3px rgba(138,99,246,.12) !important;
}

#generate {
    background: linear-gradient(100deg, #9157f6, #527ff5) !important;
    border: none !important;
    border-radius: 14px !important;
    color: white !important;
    font-size: 17px !important;
    font-weight: 700 !important;
    min-height: 54px;
    box-shadow: 0 9px 20px rgba(106, 91, 225, .23);
    transition: transform .2s ease, box-shadow .2s ease;
}

#generate:hover {
    box-shadow: 0 12px 28px rgba(106, 91, 225, .32);
}

#summary-output textarea {
    background: #f8f9ff !important;
    color: #252d5a !important;
    font-size: 16px !important;
    line-height: 1.9 !important;
}

#stats {
    border-radius: 13px;
    background: linear-gradient(100deg, #f0ebff, #edf4ff);
    padding: 12px 16px;
    color: #44358c;
}

#features {
    text-align: center;
    color: #47547d;
    padding: 24px 10px 5px;
    line-height: 2;
}

#footer {
    text-align: center;
    color: #777fa4;
    padding: 18px;
    font-size: 13px;
}

footer {
    display: none !important;
}
"""

# ----------------------------------------------------------
# 5. BUILD THE BEAUTIFUL GRADIO INTERFACE
# ----------------------------------------------------------

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="AI Text Summarizer | Hugging Face"
) as app:

    gr.HTML("""
    <div id="hero">
        <h1>🤖 AI Text Summarizer</h1>
        <p>
            Turn lengthy articles into clear, meaningful summaries.<br>
            Save time. Capture the important ideas. ✨
        </p>
        <div id="model-tag">
            🤗 Powered by Hugging Face · T5 Small
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):

        # LEFT PANEL
        with gr.Column(scale=1, elem_classes=["panel"]):

            gr.Markdown("### 📝 Enter Your Text")

            gr.Markdown(
                "Paste a long article, paragraph, or document below."
            )

            text_input = gr.Textbox(
                label="Your original text",
                placeholder=(
                    "Paste your text here...\n\n"
                    "Enter at least 20 words to get started."
                ),
                lines=12,
                max_lines=18
            )

            with gr.Row():
                word_display = gr.Markdown(
                    "**0 words entered**"
                )

            gr.Markdown("### 🎚️ Summary Length")

            max_tokens = gr.Slider(
                minimum=20,
                maximum=100,
                value=50,
                step=10,
                label="Maximum generated tokens"
            )

            summarize_button = gr.Button(
                "✨  Generate Summary",
                variant="primary",
                elem_id="generate"
            )

            clear_button = gr.ClearButton(
                components=[text_input],
                value="🗑️ Clear Text"
            )

        # RIGHT PANEL
        with gr.Column(scale=1, elem_classes=["panel"]):

            gr.Markdown("### ✨ Your AI Summary")

            gr.Markdown(
                "Your concise summary will appear in this space."
            )

            summary_output = gr.Textbox(
                label="Generated summary",
                value="Your summary will appear here...",
                lines=12,
                max_lines=18,
                interactive=False,
                elem_id="summary-output"
            )

            stats_output = gr.Markdown(
                "**📊 Word count:** Waiting for input...",
                elem_id="stats"
            )

            message_output = gr.Markdown(
                "💡 Your key ideas, simplified."
            )

            gr.Markdown(
                """
                **🤗 Model:** T5 Small
                **🧠 Task:** Abstractive text summarization
                **⚙️ Framework:** Hugging Face Transformers
                """
            )

    gr.HTML("""
    <div id="features">
        <h3>Why You'll Love It ✨</h3>
        <p>
            ⚡ Fast & Easy &nbsp;&nbsp; · &nbsp;&nbsp;
            🧠 AI Powered &nbsp;&nbsp; · &nbsp;&nbsp;
            📚 Student Friendly &nbsp;&nbsp; · &nbsp;&nbsp;
            💜 Simple to Use
        </p>
    </div>

    <div id="footer">
        Built with Python, Gradio & Hugging Face Transformers<br>
        <b>Shorter Text. Bigger Ideas.</b>
    </div>
    """)

    # Update the live word count while typing
    text_input.change(
        fn=lambda text: f"**{len(text.split())} words entered**",
        inputs=text_input,
        outputs=word_display
    )

    # Generate the summary
    summarize_button.click(
        fn=summarize_app,
        inputs=[text_input, max_tokens],
        outputs=[
            summary_output,
            stats_output,
            message_output
        ]
    )

# ----------------------------------------------------------
# 6. LAUNCH THE APPLICATION
# ----------------------------------------------------------

app.queue().launch(share=True)

⏳ Downloading the Hugging Face model...
✅ Hugging Face model loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e56ae9d3bbf6a86bcc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
